# 3D Gaussian Splatting — Google Colab 학습

이 노트북은 [hojunkim02/gaussian-orgin](https://github.com/hojunkim02/gaussian-orgin) 레포지토리(3D Gaussian Splatting)를 Google Colab GPU 환경에서 학습하기 위한 것입니다.

**사용 방법**
1. `런타임 > 런타임 유형 변경`에서 **GPU(T4 권장)**를 선택하세요.
2. 위에서부터 순서대로 셀을 실행하세요.
3. 학습 결과는 `output/`에 저장되며, 마지막 셀에서 `.zip`으로 다운로드할 수 있습니다.

`DATA_PATH`를 자신의 데이터셋 경로로 바꾸면 자신의 데이터로 학습할 수 있습니다. 전체 학습(30K 반복)은 T4 기준 수 시간 소요되므로, 테스트 시에는 `ITERATIONS`를 줄이세요.

In [ ]:
# GPU 확인
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요: 런타임 > 런타임 유형 변경 > GPU'

In [ ]:
# 레포지토리 클론 및 서브모듈 초기화
REPO_URL = "https://github.com/hojunkim02/gaussian-orgin"
%cd /content
![ -d gaussian-orgin ] || git clone --recursive $REPO_URL
%cd gaussian-orgin
!git submodule sync
!git submodule update --init --recursive --force

In [ ]:
# ==== 버전 호환성 검사 (Colab 버전 문제 진단) ====
# 이 셀은 3DGS 빌드 시 흔히 발생하는 버전 문제를 자동으로 진단합니다.
import subprocess, os, re, torch, glob, shutil, importlib.util

def run(cmd):
    try: return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception: return ""

print("== 설치된 버전 ==")
print("torch:", torch.__version__, "| torch.version.cuda:", torch.version.cuda)
nvcc_ver = run("nvcc --version")
print("nvcc:\n", nvcc_ver if nvcc_ver else "(nvcc 없음!)")
gcc_ver = run("gcc --version").splitlines()[0] if run("gcc --version") else "없음"
print("gcc:", gcc_ver)
print("GPU:", torch.cuda.get_device_name(0), "| compute capability:", torch.cuda.get_device_capability(0))
print("CUDA_HOME:", os.environ.get("CUDA_HOME"), "| which nvcc:", run("which nvcc"))
print("CUDA 툴킷 설치본:", sorted(glob.glob('/usr/local/cuda*')))
import numpy; print("numpy:", numpy.__version__)

# CUDA_HOME 이 비어 있으면 nvcc 위치에서 유추
if not os.environ.get("CUDA_HOME"):
    nvcc_path = run("which nvcc")
    if nvcc_path:
        os.environ["CUDA_HOME"] = os.path.dirname(os.path.dirname(nvcc_path))
        print("[OK] CUDA_HOME 유추:", os.environ["CUDA_HOME"])

# --- 검사 1: nvcc 툴킷 메이저 버전 vs PyTorch가 빌드된 CUDA 버전 ---
m = re.search(r"release (\d+)\.(\d+)", nvcc_ver)
nvcc_major, nvcc_minor = (int(m.group(1)), int(m.group(2))) if m else (None, None)
torch_cuda_major = int(torch.version.cuda.split('.')[0])
if nvcc_major is None:
    raise RuntimeError("nvcc를 찾을 수 없습니다. 이 셀을 재실행하거나 CUDA 툴킷 경로를 확인하세요.")
if nvcc_major != torch_cuda_major:
    print(f"\n[경고] nvcc({nvcc_major}.{nvcc_minor}) 와 torch CUDA({torch.version.cuda}) 메이저 버전 불일치!")
    cand = [d for d in glob.glob(f'/usr/local/cuda-{torch_cuda_major}*')]
    if cand:
        os.environ['CUDA_HOME'] = cand[0]
        os.environ['PATH'] = cand[0] + '/bin:' + os.environ['PATH']
        os.environ['LD_LIBRARY_PATH'] = cand[0] + '/lib64:' + os.environ.get('LD_LIBRARY_PATH','')
        print(f"-> CUDA_HOME을 {cand[0]} 로 전환했습니다.")
    else:
        raise RuntimeError(f"torch CUDA {torch.version.cuda} 에 맞는 툴킷이 없습니다.")
else:
    print(f"[OK] nvcc({nvcc_major}) 와 torch CUDA({torch_cuda_major}) 메이저 버전 일치")

# --- 검사 2: gcc 호스트 컴파일러 (CUDA 메이저 버전별 지원 상한) ---
# 지원 상한: CUDA 11.x -> gcc 10, CUDA 12.x -> gcc 12~14, CUDA 13.x -> gcc 14
gcc_major_max = {11: 10, 12: 14, 13: 14}.get(nvcc_major, 12)
gm = re.search(r"gcc.*?(\d+)\.", gcc_ver)
if gm:
    gcc_major = int(gm.group(1))
    if gcc_major > gcc_major_max:
        print(f"[경고] gcc {gcc_major} > CUDA {nvcc_major} 지원 상한({gcc_major_max})")
        alt_found = False
        for alt in range(gcc_major_max, 9, -1):
            if shutil.which(f'gcc-{alt}'):
                os.environ['CC'], os.environ['CXX'] = f'gcc-{alt}', f'g++-{alt}'
                print(f"-> CC/CXX를 gcc-{alt}/g++-{alt} 로 지정했습니다.")
                alt_found = True
                break
        if not alt_found:
            print("-> 지원되는 대체 컴파일러가 없습니다. apt 설치를 시도하세요.")
    else:
        print(f"[OK] gcc {gcc_major} 는 CUDA {nvcc_major} 지원 범위(<= {gcc_major_max}) 내")

# --- 검사 3: GPU compute capability vs TORCH_CUDA_ARCH_LIST ---
cc = torch.cuda.get_device_capability(0)
arch = f"{cc[0]}.{cc[1]}"
os.environ['TORCH_CUDA_ARCH_LIST'] = arch
print(f"[OK] TORCH_CUDA_ARCH_LIST={os.environ['TORCH_CUDA_ARCH_LIST']} (GPU compute capability {arch})")

# --- 검사 4: 빌드 중 RAM OOM 방지 ---
if importlib.util.find_spec('psutil') is None:
    !pip install -q psutil
import psutil
mem_gb = psutil.virtual_memory().total / 2**30
os.environ['MAX_JOBS'] = '2' if mem_gb < 14 else '4'
print(f"[OK] 시스템 RAM {mem_gb:.1f}GB -> MAX_JOBS={os.environ['MAX_JOBS']}")

In [ ]:
# 서브모듈 빌드 및 설치 (몇 분 소요)
# --no-build-isolation 필수: pip의 격리 빌드 환경에는 torch가 없어 setup.py가 죽음
!pip install -q plyfile tqdm opencv-python joblib ninja setuptools wheel psutil

import os, subprocess, re
DGR = 'submodules/diff-gaussian-rasterization'
GLM = f'{DGR}/third_party/glm'
GLM_COMMIT = '5c46b9c07008ae65cb81ab79cd677ecc1934b903'

# ---- CUDA 13 호환 패치 ----
# CUDA 12까지는 CUDA 헤더가 <cstdint>를 암시적으로 포함했지만 CUDA 13부터는 그렇지 않아
# uint32_t/uint64_t/std::uintptr_t 미정의 컴파일 오류가 발생한다. 빌드 전 헤더에 include 추가.
patch_targets = [f'{DGR}/cuda_rasterizer/rasterizer_impl.h',
                 f'{DGR}/cuda_rasterizer/forward.h',
                 f'{DGR}/cuda_rasterizer/backward.h']
for pth in patch_targets:
    if not os.path.isfile(pth):
        continue
    src = open(pth).read()
    if '#include <cstdint>' not in src:
        lines = src.splitlines(keepends=True)
        for k, line in enumerate(lines):
            if line.startswith('#include'):
                lines.insert(k, '#include <cstdint>\n')
                break
        open(pth, 'w').write(''.join(lines))
        print(f'[패치] {pth} — #include <cstdint> 추가')

# ---- glm 중첩 서브모듈 확보 ----
if not os.path.isfile(f'{GLM}/glm/glm.hpp'):
    print('glm 헤더 없음 — 서브모듈 재동기화')
    !git submodule update --init --recursive {DGR}
if not os.path.isfile(f'{GLM}/glm/glm.hpp'):
    print('여전히 없음 — glm을 직접 클론')
    !rm -rf {GLM}
    !git clone -q https://github.com/g-truc/glm.git {GLM}
!cd {GLM} && git checkout -q {GLM_COMMIT} 2>/dev/null || (git fetch -q origin && git checkout -q {GLM_COMMIT})
assert os.path.isfile(f'{GLM}/glm/glm.hpp'), 'glm/glm.hpp 확보 실패'
print('[OK] glm 준비 완료 (커밋', GLM_COMMIT[:8], ')')

def pip_install(pkg_path):
    """verbose 로그 저장, 실패 시 실제 nvcc/gcc 오류 라인을 화면에 출력."""
    log = f'/tmp/build_{os.path.basename(pkg_path)}.log'
    r = subprocess.run(['pip', 'install', '--no-build-isolation', '-v', pkg_path],
                       capture_output=True, text=True)
    full = r.stdout + '\n' + r.stderr
    open(log, 'w').write(full)
    if r.returncode != 0:
        err_lines = [l for l in full.splitlines()
                     if ('FAILED:' in l or 'error:' in l.lower() or 'fatal error' in l.lower())
                     and 'RuntimeError' not in l]
        print(f'==== {pkg_path} 빌드 실패! 컴파일러 오류 ====')
        print('\n'.join(err_lines[-40:]) if err_lines else '\n'.join(full.splitlines()[-60:]))
        raise RuntimeError(f'{pkg_path} 빌드 실패 — 위 오류를 확인하세요')
    print(f'[OK] {pkg_path}')

pip_install('submodules/diff-gaussian-rasterization')
pip_install('submodules/simple-knn')
pip_install('submodules/fused-ssim')

# 임포트 검증 (libcudart.so 불일치 오류는 여기서 즉시 드러남)
import diff_gaussian_rasterization, simple_knn, fused_ssim
from diff_gaussian_rasterization import GaussianRasterizer
print('모든 CUDA 확장 모듈 임포트 성공')

In [ ]:
# 데이터셋 준비
# 기본값: 공식 예제 데이터셋(tandt_db, 약 650MB) 중 truck 장면 사용
# 자신의 데이터를 쓰려면 USE_EXAMPLE_DATA=False 로 두고 DATA_PATH를 지정하세요.
DATA_PATH = ""  # @param {type:"string"}
USE_EXAMPLE_DATA = True  # @param {type:"boolean"}

if USE_EXAMPLE_DATA:
    %cd /content/gaussian-orgin
    import os
    if not os.path.isdir("data/tandt"):
        os.makedirs("data", exist_ok=True)
        !wget -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O data/tandt_db.zip
        !unzip -q data/tandt_db.zip -d data && rm data/tandt_db.zip
    DATA_PATH = "/content/gaussian-orgin/data/tandt/truck"

print("데이터 경로:", DATA_PATH)
assert os.path.isdir(DATA_PATH), f"데이터 경로가 존재하지 않습니다: {DATA_PATH}"

In [ ]:
# (선택) 구글 드라이브 마운트 — 자신의 데이터셋 로드 또는 결과 저장용
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 학습 실행
# ITERATIONS: Colab 세션 길이에 맞게 조절 (예: 테스트 7000 / 전체 30000)
ITERATIONS = 30000  # @param {type:"integer"}
SAVE_ITER = 7000    # @param {type:"integer"}
OUTPUT_DIR = "/content/gaussian-orgin/output"

%cd /content/gaussian-orgin
!python train.py -s $DATA_PATH -m $OUTPUT_DIR --iterations $ITERATIONS --save_iterations $SAVE_ITER --test_iterations $SAVE_ITER

In [ ]:
# 렌더링 (테스트/렌더 카메라 뷰)
%cd /content/gaussian-orgin
!python render.py -m $OUTPUT_DIR

In [ ]:
# (선택) 지표 계산 (PSNR / SSIM / LPIPS)
%cd /content/gaussian-orgin
!pip install -q lpips
!python metrics.py -m $OUTPUT_DIR

In [ ]:
# 렌더링 결과를 GIF로 미리보기
import glob, imageio, os
renders = sorted(glob.glob('/content/gaussian-orgin/output/train/renders/*.png'))
if renders:
    frames = [imageio.imread(p) for p in renders[:60]]
    imageio.mimsave('/content/preview.gif', frames, duration=0.1)
    from IPython.display import Image, display
    display(Image('/content/preview.gif'))
else:
    print("렌더링 결과가 없습니다. render.py 셀을 먼저 실행하세요.")

In [ ]:
# 학습 결과(모델 .ply 포함) 다운로드
import shutil
from google.colab import files
shutil.make_archive('/content/gaussian_output', 'zip', '/content/gaussian-orgin/output')
files.download('/content/gaussian_output.zip')